# TUGAS MANDIRI PRAKTIKUM BIG DATA — PERTEMUAN 3
## Pengenalan Big Data & Arsitektur HDFS (Hadoop Distributed File System)

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 3 (Tugas Mandiri) |
| **Nama Mahasiswa** | Yarsya |
| **NPM** | 2505060066 |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |

---

### Skenario Tugas
Sebagai Data Analyst junior pada platform e-commerce yang sedang bermigrasi ke arsitektur **Big Data**, kita ditugaskan membangun pipeline penyimpanan data terdistribusi menggunakan **Hadoop HDFS**. Alur kerja mencakup:
1. Membangun struktur direktori berstandar enterprise di HDFS (`raw` dan `processed`).
2. Mengunggah data transaksi mentah dari tiga cabang kota (Magelang, Yogyakarta, dan Semarang) ke direktori `raw` di HDFS.
3. Membaca kembali ketiga berkas secara langsung dari HDFS menggunakan Python (tanpa unduh ke disk lokal) dan menggabungkannya ke dalam satu DataFrame.
4. Mengolah data dengan menambahkan kalkulasi metrik `total_pendapatan`, membuat tabel ringkasan agregasi kota x kategori, menyimpan hasilnya, dan mengunggah ke direktori `processed` di HDFS.
5. Menyertakan dokumentasi NameNode Web UI dan analisis reflektif terkait pemisahan direktori data mentah vs data olahan.

## 0. Verifikasi Kesiapan Environment & Daemon Hadoop
Memastikan bahwa seluruh proses daemon Hadoop (**NameNode**, **DataNode**, dan **SecondaryNameNode**) aktif dan siap melayani operasi berkas HDFS.

In [1]:
# Verifikasi versi Hadoop
!hdfs version

Hadoop 3.4.3
Source code repository https://github.com/apache/hadoop.git -r 9d50c6884666e794e45102260a4017bb31802e1b
Compiled by stevel on 2026-02-13T14:23Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4
From source with checksum 2331238c4c2929e66645316a32a8613
This command was run using /home/meil/hadoop/share/hadoop/common/hadoop-common-3.4.3.jar


In [2]:
# Verifikasi daemon Hadoop yang sedang berjalan
!jps

23315 Jps
6020 SparkSubmit
7510 SparkSubmit
4326 SparkSubmit
2824 SecondaryNameNode
3176 NodeManager
2440 NameNode
3034 ResourceManager
2618 DataNode


## 1. Persiapan Dataset Transaksi Tiga Cabang Kota
Membangkitkan tiga dataset transaksi cabang (**Magelang**, **Yogyakarta**, **Semarang**) periode Agustus 2026 secara deterministik (menggunakan seed konsisten).

In [3]:
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


---
## Bagian A. Membangun Struktur Direktori HDFS (Bobot 15%)

Membuat struktur direktori berikut di HDFS:
```
/user/meil/ecommerce/raw
/user/meil/ecommerce/processed
```
Opsi `-p` memastikan direktori induk (`/user/meil/ecommerce`) otomatis terbuat jika belum ada.

In [4]:
# Membuat direktori raw dan processed di HDFS
!hdfs dfs -mkdir -p /user/meil/ecommerce/raw
!hdfs dfs -mkdir -p /user/meil/ecommerce/processed

# Menampilkan struktur direktori rekursif untuk membuktikan berhasil terbentuk
!hdfs dfs -ls -R /user/meil/ecommerce

drwxr-xr-x   - meil supergroup          0 2026-11-05 16:58 /user/meil/ecommerce/processed
-rw-r--r--   1 meil supergroup      41257 2026-11-05 16:58 /user/meil/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 meil supergroup        671 2026-11-05 16:58 /user/meil/ecommerce/processed/ringkasan_kota_kategori.csv
drwxr-xr-x   - meil supergroup          0 2026-11-05 16:57 /user/meil/ecommerce/raw
-rw-r--r--   1 meil supergroup      12329 2026-11-05 16:57 /user/meil/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 meil supergroup      12154 2026-11-05 16:57 /user/meil/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 meil supergroup      12684 2026-11-05 16:57 /user/meil/ecommerce/raw/transaksi_yogyakarta.csv


---
## Bagian B. Mengunggah Data Mentah ke HDFS (Bobot 15%)

Mengunggah ketiga berkas CSV (`transaksi_magelang.csv`, `transaksi_yogyakarta.csv`, `transaksi_semarang.csv`) ke direktori `/user/meil/ecommerce/raw` di HDFS menggunakan perintah `hdfs dfs -put`.

In [7]:
# Unggah ketiga berkas CSV ke HDFS direktori raw
!hdfs dfs -put -f transaksi_magelang.csv transaksi_yogyakarta.csv transaksi_semarang.csv /user/meil/ecommerce/raw/

# Tampilkan isi direktori raw sebagai bukti terunggah lengkap beserta ukurannya
!hdfs dfs -ls -h /user/meil/ecommerce/raw

Found 3 items
-rw-r--r--   1 meil supergroup     12.0 K 2026-11-19 17:35 /user/meil/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 meil supergroup     11.9 K 2026-11-19 17:35 /user/meil/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 meil supergroup     12.4 K 2026-11-19 17:35 /user/meil/ecommerce/raw/transaksi_yogyakarta.csv


In [8]:
# Tampilkan ringkasan penggunaan kapasitas disk per berkas
!hdfs dfs -du -h /user/meil/ecommerce/raw

12.0 K  12.0 K  /user/meil/ecommerce/raw/transaksi_magelang.csv
11.9 K  11.9 K  /user/meil/ecommerce/raw/transaksi_semarang.csv
12.4 K  12.4 K  /user/meil/ecommerce/raw/transaksi_yogyakarta.csv


---
## Bagian C. Membaca Kembali dan Menggabungkan Data dari HDFS (Bobot 25%)

Membaca kembali ketiga berkas **langsung dari HDFS** menggunakan Python (bukan dari disk lokal) dengan mengalirkan output perintah `hdfs dfs -cat [path]` ke memory stream `io.BytesIO`, lalu diparse langsung oleh `pandas.read_csv()`.

Setelah itu, ketiga dataset digabungkan menjadi satu DataFrame dan dibuktikan dengan menampilkan `value_counts()` pada kolom `kota`.

In [9]:
import io
import subprocess
import pandas as pd

# Path direktori HDFS data mentah
hdfs_raw_dir = "/user/meil/ecommerce/raw"
daftar_berkas = [
    "transaksi_magelang.csv",
    "transaksi_yogyakarta.csv",
    "transaksi_semarang.csv"
]

list_df = []
for nama_berkas in daftar_berkas:
    hdfs_path = f"{hdfs_raw_dir}/{nama_berkas}"
    # Membaca stream konten langsung dari HDFS via cat
    proses = subprocess.run(["hdfs", "dfs", "-cat", hdfs_path], capture_output=True, check=True)
    # Membaca bytes stream langsung ke pandas tanpa menyimpan ke disk lokal
    df_cabang = pd.read_csv(io.BytesIO(proses.stdout))
    print(f"Berhasil memuat dari HDFS: {nama_berkas} ({df_cabang.shape[0]} baris, {df_cabang.shape[1]} kolom)")
    list_df.append(df_cabang)

# Menggabungkan ketiganya menjadi satu DataFrame
df_gabungan = pd.concat(list_df, ignore_index=True)

print("\n" + "="*60)
print(f"Total baris DataFrame gabungan: {df_gabungan.shape[0]} baris")
print("="*60)

# Pembuktian hasil gabungan berisi transaksi dari ketiga kota (value_counts)
print("\nBukti Transaksi dari Ketiga Kota (value_counts):")
display(df_gabungan["kota"].value_counts().to_frame(name="total_transaksi"))

# Tampilkan 5 baris pertama data gabungan
print("\n5 Baris Teratas DataFrame Gabungan:")
display(df_gabungan.head())

Berhasil memuat dari HDFS: transaksi_magelang.csv (200 baris, 7 kolom)
Berhasil memuat dari HDFS: transaksi_yogyakarta.csv (200 baris, 7 kolom)
Berhasil memuat dari HDFS: transaksi_semarang.csv (200 baris, 7 kolom)

Total baris DataFrame gabungan: 600 baris

Bukti Transaksi dari Ketiga Kota (value_counts):


,total_transaksi
kota,
Magelang,200
Yogyakarta,200
Semarang,200



5 Baris Teratas DataFrame Gabungan:


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang


---
## Bagian D. Mengolah dan Mengunggah Hasil ke Direktori `processed` (Bobot 25%)

Langkah pengolahan data:
1. Menambahkan kolom `total_pendapatan` (`unit_terjual * harga_satuan`).
2. Membuat satu tabel ringkasan agregasi (`groupby`) yang menampilkan total pendapatan per kota dan per kategori sekaligus.
3. Menyimpan kedua hasil tersebut ke disk lokal (`data_gabungan_bersih.csv` dan `ringkasan_kota_kategori.csv`), lalu mengunggah keduanya ke `/user/meil/ecommerce/processed` di HDFS.

In [10]:
# 1. Menambahkan kolom total_pendapatan (unit_terjual x harga_satuan)
df_gabungan["total_pendapatan"] = df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]

print("5 Baris Teratas setelah penambahan kolom 'total_pendapatan':")
display(df_gabungan[["order_id", "kota", "kategori", "unit_terjual", "harga_satuan", "total_pendapatan"]].head())

5 Baris Teratas setelah penambahan kolom 'total_pendapatan':


,order_id,kota,kategori,unit_terjual,harga_satuan,total_pendapatan
0,MAG-2000,Magelang,Fashion,1,50000,50000
1,MAG-2001,Magelang,Elektronik,7,25000,175000
2,MAG-2002,Magelang,Elektronik,7,25000,175000
3,MAG-2003,Magelang,Rumah Tangga,6,25000,150000
4,MAG-2004,Magelang,Fashion,7,100000,700000


In [11]:
# 2. Membuat tabel ringkasan (groupby) total pendapatan per kota dan per kategori
ringkasan_kota_kategori = df_gabungan.groupby(["kota", "kategori"]).agg(
    total_unit_terjual=("unit_terjual", "sum"),
    total_pendapatan=("total_pendapatan", "sum"),
    jumlah_transaksi=("order_id", "count")
).reset_index()

# Urutkan agar tampilan rapi
ringkasan_kota_kategori = ringkasan_kota_kategori.sort_values(by=["kota", "total_pendapatan"], ascending=[True, False])

print("Tabel Ringkasan Pendapatan per Kota dan Kategori:")
display(ringkasan_kota_kategori)

Tabel Ringkasan Pendapatan per Kota dan Kategori:


,kota,kategori,total_unit_terjual,total_pendapatan,jumlah_transaksi
1,Magelang,Fashion,230,27750000,52
0,Magelang,Elektronik,197,18775000,45
3,Magelang,Makanan & Minuman,144,17525000,39
2,Magelang,Kesehatan & Kecantikan,138,17375000,33
4,Magelang,Rumah Tangga,123,12200000,31
6,Semarang,Fashion,219,26425000,55
5,Semarang,Elektronik,222,21425000,52
8,Semarang,Makanan & Minuman,142,19750000,39
7,Semarang,Kesehatan & Kecantikan,114,13525000,26
9,Semarang,Rumah Tangga,115,10975000,28


In [12]:
# 3. Menyimpan kedua hasil sebagai berkas CSV baru di disk lokal
file_bersih = "data_gabungan_bersih.csv"
file_ringkasan = "ringkasan_kota_kategori.csv"

df_gabungan.to_csv(file_bersih, index=False)
ringkasan_kota_kategori.to_csv(file_ringkasan, index=False)

print(f"Berkas lokal '{file_bersih}' berhasil disimpan ({df_gabungan.shape[0]} baris).")
print(f"Berkas lokal '{file_ringkasan}' berhasil disimpan ({ringkasan_kota_kategori.shape[0]} baris).")

Berkas lokal 'data_gabungan_bersih.csv' berhasil disimpan (600 baris).
Berkas lokal 'ringkasan_kota_kategori.csv' berhasil disimpan (15 baris).


In [13]:
# Mengunggah kedua berkas hasil olahan ke HDFS direktori processed
!hdfs dfs -put -f data_gabungan_bersih.csv ringkasan_kota_kategori.csv /user/meil/ecommerce/processed/

# Tampilkan isi direktori processed sebagai bukti keberhasilan unggah
!hdfs dfs -ls -h /user/meil/ecommerce/processed

Found 2 items
-rw-r--r--   1 meil supergroup     40.3 K 2026-11-19 17:35 /user/meil/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 meil supergroup        671 2026-11-19 17:35 /user/meil/ecommerce/processed/ringkasan_kota_kategori.csv


In [14]:
# Menampilkan isi berkas ringkasan_kota_kategori.csv langsung dari HDFS
!hdfs dfs -cat /user/meil/ecommerce/processed/ringkasan_kota_kategori.csv

kota,kategori,total_unit_terjual,total_pendapatan,jumlah_transaksi
Magelang,Fashion,230,27750000,52
Magelang,Elektronik,197,18775000,45
Magelang,Makanan & Minuman,144,17525000,39
Magelang,Kesehatan & Kecantikan,138,17375000,33
Magelang,Rumah Tangga,123,12200000,31
Semarang,Fashion,219,26425000,55
Semarang,Elektronik,222,21425000,52
Semarang,Makanan & Minuman,142,19750000,39
Semarang,Kesehatan & Kecantikan,114,13525000,26
Semarang,Rumah Tangga,115,10975000,28
Yogyakarta,Fashion,249,26625000,66
Yogyakarta,Kesehatan & Kecantikan,138,19700000,32
Yogyakarta,Elektronik,158,18500000,40
Yogyakarta,Rumah Tangga,123,17275000,30
Yogyakarta,Makanan & Minuman,138,15175000,32


---
## Bagian E. Dokumentasi dan Refleksi (Bobot 20%)

### 1. Dokumentasi NameNode Web UI
Berikut dokumentasi tangkapan layar (*screenshot*) antarmuka NameNode Web UI (http://localhost:9870, menu *Utilities → Browse the file system*) yang membuktikan struktur direktori `/user/meil/ecommerce/` beserta seluruh isinya di HDFS.

#### A. Struktur Folder `/user/meil/ecommerce/` (`raw` dan `processed`)
![Struktur Folder Ecommerce](screenshot_hdfs_ecommerce.png)

#### B. Isi Folder `/user/meil/ecommerce/raw/` (Data Mentah Cabang)
![Isi Folder Raw](screenshot_hdfs_raw.png)

#### C. Isi Folder `/user/meil/ecommerce/processed/` (Data Bersih & Ringkasan)
![Isi Folder Processed](screenshot_hdfs_processed.png)

### 2. Tulisan Reflektif

**Soal Refleksi:**
*Apa keuntungan menyimpan data mentah (raw) terpisah dari data olahan (processed) di HDFS, dibandingkan menyimpan semuanya bercampur dalam satu folder?*

**Jawaban Reflektif:**

Dalam arsitektur data terdistribusi dan *Data Lake* modern (seperti *Medallion Architecture*), memisahkan data mentah (*raw*) dengan data olahan (*processed*) ke dalam direktori independen di HDFS memberikan keuntungan strategis yang sangat krusial, antara lain:

1. **Integritas Sumber Data Asli (*Immutability & Single Source of Truth*):**
   Data mentah yang diterima langsung dari cabang transaksi bersifat permanen dan tidak boleh dimanipulasi (*immutable*). Dengan mengisolasi folder `raw`, keaslian data historis terlindungi dari risiko modifikasi yang tidak disengaja, korupsi berkas, maupun tertimpa (*overwritten*) saat proses transformasi berlangsung. Apabila suatu saat terdapat kekeliruan logika bisnis (misalnya kesalahan rumus pendapatan atau klasifikasi kategori) atau sistem memerlukan fitur agregasi baru, tim data dapat dengan mudah melakukan komputasi ulang (*re-processing/backfilling*) langsung dari sumber data aslinya tanpa khawatir kehilangan data dasar.

2. **Tata Kelola Keamanan dan Pembatasan Akses (*Data Governance & Access Control*):**
   Pada infrastruktur enterprise, izin akses berkas pada level HDFS (*POSIX permissions & ACLs*) dapat dikonfigurasi secara spesifik per folder. Direktori `raw` umumnya memiliki tingkat kerahasiaan lebih tinggi karena berpotensi memuat informasi sensitif atau atribut privat pelanggan yang belum disamarkan, sehingga akses dibatasi hanya untuk *data engineer* atau pipeline ingestion otomatis. Sebaliknya, direktori `processed` berisi data yang telah dibersihkan, divalidasi, dan diagregasi, sehingga aman dibuka untuk *data analyst*, pemodelan *machine learning*, maupun integrasi dasbor *Business Intelligence* (BI).

3. **Efisiensi Pemrosesan Komputasi dan Pencegahan Ambiguitas Pipeline:**
   Pencampuran seluruh berkas dalam satu folder berisiko menimbulkan kebingungan sistem pemrosesan batch (seperti Apache Spark atau Hadoop MapReduce), yang sering kali menggunakan pola pencarian berkas (*path globbing*, misalnya `*.csv`). Pemisahan folder memastikan proses otomatisasi membaca data dari sumber yang benar tanpa membaca ulang data yang sudah diproses secara ganda (*accidental double-processing*). Selain itu, kebijakan siklus hidup data (*data retention policy*) dapat diterapkan secara fleksibel, misalnya data di folder `raw` diarsipkan atau dikompresi setelah periode tertentu, sementara data di `processed` tetap siap digunakan untuk kueri analitik harian.